# Apply SENSE to your own Brightway database

This tutorial uses the public Python API. It does not depend on the hydrogen study.
Follow the repository installation and input-preparation instructions first. You need
an existing Brightway project, a technosphere database, a PSILCA .zolca backup,
and the ecoinvent Database Overview workbook matching your inventory version.

**Running SENSE writes social exchanges to the selected database and registers social
flows and LCIA methods.** Use a project you intend to modify. Enter your own paths and
names below. Database imports and a full manuscript reproduction are separate workflows.

In [ ]:
from pathlib import Path

import bw2data as bd

from sense import SenseConfig, run_sense

PROJECT = "my-brightway-project"
DATABASE = "ecoinvent-3.12-cutoff"
PSILCA_PATH = Path("/path/to/psilca.zolca")
OVERVIEW = Path("/path/to/Database-Overview-for-ecoinvent-v3.12.xlsx")
OUTPUT = Path("sense-results")

if PROJECT not in bd.projects:
    raise ValueError("Choose an existing Brightway project before running this tutorial")
bd.projects.set_current(PROJECT)
if DATABASE not in bd.databases:
    raise ValueError(f"Database {DATABASE!r} is absent from {PROJECT!r}")

## Choose and record your modelling assumptions

The example uses geometric aggregation and GDP country weights. These are modelling
choices, not guarantees of representative production shares or measured harm.
See the README for other settings, custom databases and external productivity inputs.
For a linked foreground, map every contributing background as well before interpreting
supply-chain social scores. Each database is passed to `run_sense` separately.

In [ ]:
config = SenseConfig(
    social_flow_mean="geometric",
    country_weighting="gdp",
)
config

## Map social data

The first run can take substantial time and downloads the sentence-transformer model.
Missing required prices or location mappings produce worklists to resolve before rerunning.
Keep the output directory: its completed run records retain configuration, provenance,
monetary assumptions and diagnostics.

In [ ]:
results = run_sense(
    project=PROJECT,
    technosphere_db=DATABASE,
    sociosphere_db="sociosphere",
    psilca_path=PSILCA_PATH,
    ecoinvent_version="3.12",
    psilca_version="4.0",
    ecoinvent_geographies_excel=OVERVIEW,
    data_dir=OUTPUT,
    config=config,
)

## Inspect diagnostics before calculating scores

Resolve processing issues and review source omissions, low-confidence sector matches,
price inference and country-weight coverage in each returned `run_directory`.
The run retains its own diagnostics, assumptions and input evidence.
A completed run is not evidence of complete or reliable indicator data.

In [ ]:
if not results or any(row.get("issues") for row in results):
    raise RuntimeError("Review and resolve SENSE processing issues before using the results")
for row in results:
    print(row)

worker_methods = [
    method
    for method in bd.methods
    if method[:3] == ("SENSE", "sociosphere", "Workers") and method[-1] == "risk"
]
worker_methods[:10]

## Evaluate the same functional unit with separate methods

Select an activity and two complete method keys from your project. The environmental
method must match its biosphere; select a worker-risk indicator only after reviewing
its source data and coverage. Set `AMOUNT` in the activity's reference-product unit.
The results have different units and must be interpreted separately. Risk-weighted
working hours describe modelled exposure to country-sector risk conditions, not
observed injuries, measured social impacts or a complete sustainability assessment.

In [ ]:
ACTIVITY_CODE = "replace-with-your-activity-code"
AMOUNT = 1.0
ENVIRONMENTAL_METHOD = ()  # Copy a complete environmental method tuple from bd.methods.
SOCIAL_METHOD = ()  # Copy a reviewed worker-risk method tuple from worker_methods.

if ENVIRONMENTAL_METHOD not in bd.methods or SOCIAL_METHOD not in worker_methods:
    raise ValueError("Choose both method keys before calculating")
activity = bd.get_activity((DATABASE, ACTIVITY_CODE))
print(f"Functional unit: {AMOUNT} {activity['unit']} of {activity['reference product']}")

In [ ]:
from bw2calc import LCA

lca = LCA({activity.key: AMOUNT}, ENVIRONMENTAL_METHOD)
lca.lci()
lca.lcia()
print("Environmental:", lca.score, bd.methods[ENVIRONMENTAL_METHOD].get("unit"))
lca.switch_method(SOCIAL_METHOD)
lca.lcia()
print("Social risk:", lca.score, bd.methods[SOCIAL_METHOD].get("unit"))